Setting Up the Environment

In [1]:
!pip install llama-index-llms-openai
!pip install -U llama-index-readers-file
!pip install llama-index-embeddings-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.1/376.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.0.0
    Uninstalling tenacity-9.0.0:
      Successfully uninstalled tenacity-9.0.0
  Attempting uninstall: nltk
    Found existing installation: nltk 3.8.1
    Uninstalling nltk-3.8.1:
      Successfully uninstalled nl

In [1]:
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
)

from llama_index.core.tools import QueryEngineTool, ToolMetadata

In [25]:
try:
    storage_context = StorageContext.from_defaults(
        persist_dir="./storage/Auditor_NAF23"
    )
    auditor_index = load_index_from_storage(storage_context)

    index_loaded = True
except:
    index_loaded = False

In [26]:
api_key=""
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

# global
Settings.embed_model = OpenAIEmbedding(api_key=api_key)

Indexing process

In [28]:
if not index_loaded:
    # load data
    auditor_docs = SimpleDirectoryReader(
        # input_files=["/content/iccrom.pdf"]
        input_files=["/content/naf23.pdf"]
    ).load_data()

    # build index
    auditor_index= VectorStoreIndex.from_documents(auditor_docs)
    # persist index

    auditor_index.storage_context.persist(persist_dir="./storage/Auditor_NAF23")

In [40]:
from llama_index.llms.openai import OpenAI
import os
import openai
os.environ["OPENAI_API_KEY"] = api_key
openai.api_key = os.environ["OPENAI_API_KEY"]
# llm = OpenAI(temperature=0.9, model="gpt-4-turbo")
llm = OpenAI(temperature=0.9, model=("gpt-4o"))


In [30]:
auditor_engine=auditor_index.as_query_engine(similarity_top_k=3)

In [31]:
query_engine_tools = [
    QueryEngineTool(
        query_engine=auditor_engine,
        metadata=ToolMetadata(
            name="search",
            description=(
                "test"
            ),
        ),
    ),
]

In [32]:
#from llama_index.core.agent import ReActAgent
from llama_index.core.agent.legacy.react.base import ReActAgent
from llama_index.llms.openai import OpenAI

In [41]:
context = """\
You are an expert on financial statement and auditor report, analysing balance sheet.
 """
agent = ReActAgent.from_tools(
    query_engine_tools,
    llm=llm,
    verbose=True,
    context=context
)

Run Query

In [13]:
response = agent.chat("Can you explain the reasons for the ICCROM's revenue decrease from 8.166.474 in FYE 2019 to 7.270.220 in FYE 2020")
print(str(response))

Thought: The current language of the user is English. I need to use a tool to help me answer the question.
Action: search
Action Input: {'input': "reasons for ICCROM's revenue decrease from FYE 2019 to FYE 2020"}
Observation: The decrease in ICCROM's revenue from FYE 2019 to FYE 2020 can be attributed to various factors such as a decrease in interest received on bank accounts, a decrease in gains on investment funds not realized, and an increase in interest paid. Additionally, there was a decrease in surplus for the year and a decrease in transfers from reserves, which could have contributed to the overall revenue decrease.
Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: The decrease in ICCROM's revenue from FYE 2019 to FYE 2020 was influenced by several financial factors. These include a reduction in interest received on bank accounts and a decrease in unrealized gains on investment funds. Additionally, there was an increase in inter

In [14]:
response = agent.chat("Can you analyze ICCROM's revenue performance from 2019 to 2020")
print(str(response))

Thought: To analyze ICCROM's revenue performance from 2019 to 2020, I need to look at the specific financial data and changes in revenue streams during this period.
Action: search
Action Input: {'input': 'ICCROM revenue performance analysis 2019 to 2020'}
Observation: The cash flow statement shows that ICCROM had a significant increase in net cash flows from operating activities from 2019 to 2020. In 2019, the net cash flows from operating activities were 1,182,356, while in 2020, they increased to 1,791,078. This indicates an improvement in ICCROM's revenue performance from 2019 to 2020.
Thought: With the information provided, I can analyze ICCROM's revenue performance from 2019 to 2020 without using any more tools.
Answer: The increase in net cash flows from operating activities from 1,182,356 in 2019 to 1,791,078 in 2020 suggests that ICCROM experienced an improvement in its operational efficiency and revenue management. This improvement in cash flow indicates that despite a decreas

In [22]:
response = agent.chat("Can you analyze and check what caused the difference in ICCROM's expsense from 2019 to 2020")
print(str(response))

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: search
Action Input: {'input': 'causes for changes in ICCROM expenses from 2019 to 2020'}
Observation: Increase in employee benefits, provisions, and interest paid were the main causes for changes in ICCROM expenses from 2019 to 2020.
Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: The primary factors that caused changes in ICCROM's expenses from 2019 to 2020 include an increase in employee benefits, provisions, and interest paid. These changes suggest that ICCROM may have adjusted its staffing policies, perhaps increasing salaries or benefits, set aside additional funds for future liabilities or contractual obligations, and faced higher costs associated with borrowing or financial management. Each of these factors would directly impact the financial statements, reflecting a strategic response to internal needs or extern

In [34]:
response = agent.chat("Can you analyse National Ataxia Foudnations's revenue change from FYE 2022 to FYE 2023")
print(str(response))

Thought: The current language of the user is English. I need to use a tool to search for information about the revenue change of the National Ataxia Foundation from FYE 2022 to FYE 2023.
Action: search
Action Input: {'input': 'National Ataxia Foundation revenue change from FYE 2022 to FYE 2023'}
Observation: The National Ataxia Foundation's revenue decreased by $2,709 from FYE 2022 to FYE 2023.
Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: The National Ataxia Foundation's revenue decreased by $2,709 from FYE 2022 to FYE 2023.
The National Ataxia Foundation's revenue decreased by $2,709 from FYE 2022 to FYE 2023.


In [36]:
response = agent.chat("Can you analyse National Ataxia Foudnations's current liabilities and find any increased items")
print(str(response))

Thought: The current language of the user is English. I need to use the search tool to find information on the National Ataxia Foundation's current liabilities and identify any items that have increased from a previous fiscal year.
Action: search
Action Input: {'input': 'National Ataxia Foundation current liabilities increase FYE 2023'}
Observation: National Ataxia Foundation's current liabilities increased at the end of the fiscal year 2023.
Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: The National Ataxia Foundation's current liabilities have increased by the end of the fiscal year 2023.
The National Ataxia Foundation's current liabilities have increased by the end of the fiscal year 2023.


In [42]:
response = agent.chat("Can you analyse the composition National Ataxia Foudnations's current assets, provide detailed reason and drivers for any decrease")
print(str(response))

Thought: The current language of the user is English. I need to use a tool to help me answer the question.
Action: search
Action Input: {'input': 'National Ataxia Foundation current assets composition 2023'}
Observation: Cash and cash equivalents: $1,008,716  
Accounts receivable: $6,438  
Prepaid expenses: $255,729
Thought: I have the current assets of National Ataxia Foundation, which include cash and cash equivalents, accounts receivable, and prepaid expenses. Now, I need to determine the detailed reasons and drivers for any decrease in these assets.
Action: search
Action Input: {'input': 'reasons for decrease in assets for National Ataxia Foundation 2023'}
Observation: The decrease in assets for the National Ataxia Foundation in 2023 could be attributed to various factors such as increased expenses, investment losses, or net assets being released from restrictions.
Thought: I now have information about the possible reasons for the decrease in assets for the National Ataxia Foundati

In [43]:
response = agent.chat("Can you analyse what caused the decrease in National Ataxia Foudnations's current assets, provide detailed defintions and drivers")
print(str(response))

Thought: The current language of the user is English. I need to use a tool to gather detailed financial information about the National Ataxia Foundation's current assets and the reasons for any decrease.
Action: search
Action Input: {'input': 'National Ataxia Foundation current assets decrease analysis 2023'}
Observation: The National Ataxia Foundation's current assets decreased in 2023 compared to 2022.
Thought: I will provide a detailed analysis of the potential reasons and drivers behind the decrease in the National Ataxia Foundation's current assets in 2023 compared to 2022.

---

### Definitions: 

1. **Current Assets**:
   - **Cash and Cash Equivalents**: These include cash on hand, demand deposits, and short-term investments that are readily convertible to cash.
   - **Accounts Receivable**: Money owed to the foundation by donors, partners, or other entities.
   - **Prepaid Expenses**: Costs that have been paid in advance for services or goods to be received in the future.

### 

In [44]:
response = agent.chat("Can you analyse the composition of National Ataxia Foudnations's current assets, provide defintions provided in the auditor notes for potential drivers")
print(str(response))

Thought: The user is asking for a detailed analysis of the National Ataxia Foundation's current assets including definitions and potential drivers as provided in the auditor's notes. I need to search for the auditor's report or relevant financial documents of the National Ataxia Foundation to provide accurate information.
Action: search
Action Input: {'input': 'National Ataxia Foundation auditor report 2023'}
Observation: The auditor's report for the National Ataxia Foundation for the year 2023 states that the financial statements present fairly, in all material respects, the financial position of the Foundation as of December 31, 2023, and the changes in its net assets and cash flows for the year ended, in accordance with accounting principles generally accepted in the United States of America. The audit was conducted in accordance with auditing standards generally accepted in the United States of America, and the auditor obtained sufficient and appropriate evidence to provide a basis